# PRISM-Seq v3 — §4 bar completion: diagnostics + char-LM
Clones the secret gist (now incl. `gpu_diag.py` + `gpu_charlm.py`), reconstructs `seq/`, mounts Drive, then runs the two remaining attention-diagnostic legs (Induction + Selective-Copy) and the char-LM leg. Results stream crash-safe to `MyDrive/prism_results/gpu_diag.json` and `gpu_charlm.json` (resumable).

In [ ]:
# 1) Clone the (updated) secret gist and reconstruct the seq/ package
!rm -rf /content/prism && git clone -q https://gist.github.com/300fa3eeb2c8e5ad6c1e0264565cb7ee.git /content/prism
import os, shutil
os.chdir('/content/prism')
os.makedirs('seq', exist_ok=True)
for m in ['prism_seq', 'delta', 'transformer', 'common', 'tasks']:
    if os.path.exists(m + '.py'):
        shutil.move(m + '.py', 'seq/' + m + '.py')
open('seq/__init__.py', 'w').close()
import torch
print('files:', sorted(os.listdir('.')))
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# 2) Mount Drive (crash-safe, resumable results)
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/prism_results', exist_ok=True)
print('Drive mounted')

In [ ]:
# 3a) Diagnostic legs: Induction + Selective-Copy (param-matched TF vs PRISM-quad2 vs none, 3 seeds)
!cd /content/prism && PRISM_RESULTS=/content/drive/MyDrive/prism_results python -u gpu_diag.py induction selcopy

In [ ]:
# 3b) Char-LM leg: tiny-shakespeare, param-matched PRISM-quad2 vs TF, test BPC (2 seeds)
!cd /content/prism && PRISM_RESULTS=/content/drive/MyDrive/prism_results python -u gpu_charlm.py